# 🧹 Data Cleaning — JobStreet Raw Dataset
**Columns:** Country, Title, Company, Location, Salary, Posted, Employment Type, Industry, URL  
**Raw size:** ~122,767 rows | 9 columns

This is a markdown cell that serves as the title and provides an overview of the dataset being cleaned. It lists the columns present in the raw data and its approximate size.

## Cell 1 — Install & Import Libraries

This markdown cell acts as a heading for the first code cell, indicating the purpose of the following block of code: installing and importing necessary libraries.

In [1]:
!pip install pandas numpy matplotlib seaborn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re, os

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)

print('✅ Libraries loaded')

✅ Libraries loaded


This code cell installs `pandas`, `numpy`, `matplotlib`, and `seaborn` using `pip`, and then imports these libraries along with `re` and `os`. It also sets display options for pandas DataFrames to show all columns and a wider column width, and prints a confirmation message once the libraries are loaded.

## Cell 2 — Load Raw CSV

This markdown cell is a heading for the section dedicated to loading the raw CSV file.

In [2]:
RAW_PATH     = 'jobstreet_raw.csv'   # update path if needed
CLEANED_PATH = 'jobstreet_cleaned.csv'

df = pd.read_csv(RAW_PATH, encoding='utf-8', low_memory=False)

print(f'[RAW SHAPE] → {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head(3)

[RAW SHAPE] → 28,450 rows × 9 columns


,Country,Title,Company,Location,Salary,Posted,Employment Type,Industry,URL
0,Malaysia,Assistant Jewellery Sales Manager (Malaysia) 珠寶銷售副經理 (馬來西亞),Luk Fook Prestige (Malaysia) Sdn Bhd,Kuala Lumpur,NaN,30d+ ago•Expiring,Full time,Sales,https://my.jobstreet.com/job/88167357?type=standard&ref=search-standalone&or...
1,Malaysia,Cashier (Malaysia) 出納員 (馬來西亞),Luk Fook Prestige (Malaysia) Sdn Bhd,Kuala Lumpur,NaN,30d+ ago,Full time,Retail,https://my.jobstreet.com/job/88167269?type=standard&ref=search-standalone&or...
2,Malaysia,Assistant Jewellery Sales Supervisor (Malaysia) 珠寶銷售副主任 (馬來西亞),Luk Fook Prestige (Malaysia) Sdn Bhd,Kuala Lumpur,NaN,30d+ ago•Expiring,Full time,Retail,https://my.jobstreet.com/job/88167220?type=standard&ref=search-standalone&or...


This code cell defines file paths for the raw and cleaned datasets. It then loads the `jobstreet_raw.csv` file into a pandas DataFrame named `df`, prints its shape (number of rows and columns), and displays the first three rows to give a quick overview of the data.

## Cell 3 — Initial Inspection

This markdown cell serves as a heading for the initial inspection of the loaded data.

In [3]:
print('=== Data Types ===')
print(df.dtypes)

print('\n=== Missing Values ===')
missing = df.isnull().sum()
pct     = (missing / len(df) * 100).round(2)
print(pd.DataFrame({'Missing': missing, 'Pct (%)': pct}))

print(f'\n=== Duplicate Rows ===\nDuplicates: {df.duplicated().sum():,}')

=== Data Types ===
Country            object
Title              object
Company            object
Location           object
Salary             object
Posted             object
Employment Type    object
Industry           object
URL                object
dtype: object

=== Missing Values ===
                 Missing  Pct (%)
Country                0     0.00
Title                  0     0.00
Company                0     0.00
Location               0     0.00
Salary             13397    47.09
Posted                 0     0.00
Employment Type       32     0.11
Industry            5643    19.83
URL                    0     0.00

=== Duplicate Rows ===
Duplicates: 555


This code cell performs an initial inspection of the DataFrame `df`. It prints the data types of all columns, calculates and displays the number and percentage of missing values for each column, and reports the count of duplicate rows.

## Cell 4 — Standardise Column Names

This markdown cell indicates the start of the column name standardization process.

In [4]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_', regex=False)
)
print('Columns:', df.columns.tolist())

Columns: ['country', 'title', 'company', 'location', 'salary', 'posted', 'employment_type', 'industry', 'url']


This code cell standardizes the DataFrame's column names by stripping whitespace, converting them to lowercase, and replacing spaces with underscores. Finally, it prints the list of new column names.

## Cell 5 — Drop Full Duplicates

This markdown cell is a heading for the section that handles the removal of duplicate rows.

In [5]:
before = len(df)
df = df.drop_duplicates()
df = df.reset_index(drop=True)
print(f'Removed {before - len(df):,} duplicate rows → {len(df):,} remaining')

Removed 555 duplicate rows → 27,895 remaining


This code cell removes duplicate rows from the DataFrame `df`. It first stores the original number of rows, then drops duplicates and resets the DataFrame index. It reports the number of duplicate rows removed and the remaining number of rows.

## Cell 6 — Clean `title` (Job Title)

This markdown cell introduces the cleaning steps for the `title` (job title) column.

In [6]:
df['title'] = (
    df['title']
    .astype(str)
    .str.strip()
    # Collapse multiple spaces
    .str.replace(r'\s+', ' ', regex=True)
    # Remove non-printable chars (keep unicode letters, digits, punctuation)
    .str.replace(r'[\x00-\x1f\x7f]', '', regex=True)
)
df.loc[df['title'].isin(['', 'nan']), 'title'] = np.nan
print(f"title nulls: {df['title'].isna().sum():,}")
print(df['title'].head(5))

title nulls: 0
0       Assistant Jewellery Sales Manager (Malaysia) 珠寶銷售副經理 (馬來西亞)
1                                     Cashier (Malaysia) 出納員 (馬來西亞)
2    Assistant Jewellery Sales Supervisor (Malaysia) 珠寶銷售副主任 (馬來西亞)
3                                         Big Data Hadoop Developer
4               Jewellery Sales Consultant (Malaysia) 珠寶銷售顧問 (馬來西亞)
Name: title, dtype: object


This code cell cleans the `title` column by converting it to string type, stripping whitespace, collapsing multiple internal spaces, and removing non-printable characters. It then replaces empty strings or 'nan' strings with actual `NaN` values and prints the count of nulls and the first five cleaned titles.

## Cell 7 — Clean `company`

This markdown cell acts as a heading for the cleaning process of the `company` column.

In [7]:
df['company'] = (
    df['company']
    .astype(str)
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
)
df.loc[df['company'].isin(['', 'nan']), 'company'] = np.nan
df['company'] = df['company'].fillna('Unknown')
print(f"Distinct companies: {df['company'].nunique():,}")
print(df['company'].value_counts().head(10))

Distinct companies: 11,342
company
Private Advertiser                             1216
SmartHire by SEEK                               509
RHB Banking Group                               113
Watson's Personal Care Stores Sdn. Bhd.         113
Celestica                                       107
Berjaya Starbucks Coffee Company Sdn Bhd        100
PERSOL Workforce Solutions Malaysia Sdn Bhd      90
Talent Trader Group Pte Ltd                      86
Jabil Circuit Sdn Bhd                            83
Elitez Pte Ltd                                   82
Name: count, dtype: int64


This code cell cleans the `company` column by converting it to string, stripping whitespace, and collapsing multiple internal spaces. It replaces empty strings or 'nan' strings with `NaN` and then fills any remaining `NaN` values with 'Unknown'. Finally, it prints the number of distinct companies and the top 10 most frequent company names.

## Cell 8 — Clean `location`

This markdown cell introduces the cleaning steps for the `location` column.

In [8]:
df['location'] = (
    df['location']
    .astype(str)
    .str.strip()
    .str.title()
    .str.replace(r'\s+', ' ', regex=True)
)
df.loc[df['location'].isin(['', 'Nan']), 'location'] = np.nan
df['location'] = df['location'].fillna('Unknown')
print(f"Distinct locations: {df['location'].nunique():,}")
print(df['location'].value_counts().head(10))

Distinct locations: 727
location
Kuala Lumpur                4038
Petaling Jaya               1403
Johor Bahru                 1173
Kuala Lumpur City Centre     963
Shah Alam                    873
Penang                       787
Selangor                     764
Klang/Port Klang             572
Bayan Lepas                  520
Subang Jaya                  495
Name: count, dtype: int64


This code cell cleans the `location` column by converting it to string, stripping whitespace, converting to title case, and collapsing multiple internal spaces. It replaces empty strings or 'Nan' strings with `NaN` and fills any remaining `NaN` values with 'Unknown'. It then prints the number of distinct locations and the top 10 most frequent locations.

## Cell 9 — Clean `employment_type`

This markdown cell serves as a heading for the cleaning process of the `employment_type` column.

In [9]:
df['employment_type'] = (
    df['employment_type']
    .astype(str)
    .str.strip()
    .str.title()
)
df.loc[df['employment_type'].isin(['', 'Nan']), 'employment_type'] = np.nan
df['employment_type'] = df['employment_type'].fillna('Unknown')

print(df['employment_type'].value_counts())

employment_type
Full Time     25807
Contract       1706
Part Time       331
Unknown          32
Internship       17
Temporary         2
Name: count, dtype: int64


This code cell cleans the `employment_type` column by converting it to string, stripping whitespace, and converting to title case. It replaces empty strings or 'Nan' strings with `NaN` and fills any remaining `NaN` values with 'Unknown'. Finally, it prints the value counts for each employment type.

## Cell 10 — Clean `industry`

This markdown cell introduces the cleaning steps for the `industry` column.

In [10]:
df['industry'] = (
    df['industry']
    .astype(str)
    .str.strip()
    .str.title()
    .str.replace(r'\s+', ' ', regex=True)
)
df.loc[df['industry'].isin(['', 'Nan']), 'industry'] = np.nan
df['industry'] = df['industry'].fillna('Unknown')

print(f"Distinct industries: {df['industry'].nunique():,}")
print(df['industry'].value_counts().head(15))

Distinct industries: 17
industry
Unknown            5531
Engineering        4002
Manufacturing      3730
Sales              3492
Accounting         1729
Retail             1535
Marketing          1416
Finance            1080
Healthcare         1062
Construction       1051
Human Resources    1007
Banking             812
Education           645
Design              495
Legal               193
Name: count, dtype: int64


This code cell cleans the `industry` column by converting it to string, stripping whitespace, converting to title case, and collapsing multiple internal spaces. It replaces empty strings or 'Nan' strings with `NaN` and fills any remaining `NaN` values with 'Unknown'. It then prints the number of distinct industries and the top 15 most frequent industries.

## Cell 11a — Normalise N/A Values in `salary`
57% of listings don't publish salary — that's normal for JobStreet.  
This cell converts literal `'n/a'`, `'N/A'`, `'-'`, `'none'` strings to proper `NaN` before parsing,  
and adds a clear `salary_disclosed` flag so you can filter/analyse disclosed vs undisclosed separately.

This markdown cell explains the next step: normalizing 'N/A' values in the `salary` column. It highlights that about 57% of listings do not publish salary information and introduces the idea of adding a `salary_disclosed` flag.

In [11]:
# ── Step 1: Normalise all 'n/a'-like strings → NaN ──────────────────────────
NA_STRINGS = {'n/a', 'na', 'none', 'null', '-', '--', 'nan', '', 'not specified', 'undisclosed'}

df['salary'] = df['salary'].astype(str).str.strip()
df.loc[df['salary'].str.lower().isin(NA_STRINGS), 'salary'] = np.nan

# ── Step 2: Add a disclosure flag ────────────────────────────────────────────
df['salary_disclosed'] = df['salary'].notna()

# ── Step 3: Summary ──────────────────────────────────────────────────────────
total        = len(df)
has_salary   = df['salary_disclosed'].sum()
no_salary    = total - has_salary

print(f'Total rows        : {total:,}')
print(f'Salary disclosed  : {has_salary:,}  ({has_salary/total*100:.1f}%)')
print(f'Salary NOT shown  : {no_salary:,}  ({no_salary/total*100:.1f}%)  ← normal for job boards')
print()
print('Note: salary_min / salary_max will be NaN for undisclosed rows.')
print('      Use salary_disclosed=True to filter to rows with known salary.')

Total rows        : 27,895
Salary disclosed  : 14,767  (52.9%)
Salary NOT shown  : 13,128  (47.1%)  ← normal for job boards

Note: salary_min / salary_max will be NaN for undisclosed rows.
      Use salary_disclosed=True to filter to rows with known salary.


This code cell normalizes various 'N/A'-like strings in the `salary` column to `np.nan`. It then creates a new boolean column `salary_disclosed` to indicate whether salary information is available. Finally, it provides a summary of salary disclosure, showing the total rows, and counts/percentages of disclosed and undisclosed salaries.

## Cell 11b — Clean `salary`
Parse min/max salary (MYR/month) from strings like `RM 3,000 – RM 5,000 per month`.

This markdown cell provides a heading and description for the main salary cleaning process, which involves parsing minimum and maximum salary values from complex strings.

In [12]:
def parse_salary(val):
    """
    Parse salary strings covering: MYR (RM/MYR), SGD, IDR (Rp), PHP (₱),
    THB (฿), USD, HKD, AUD — plus shorthand like '5k', 'p.m.', 'p.a.'
    Returns: salary_min, salary_max, salary_currency, salary_period
    """
    if pd.isna(val):
        return pd.Series([np.nan, np.nan, np.nan, np.nan])

    raw = str(val).strip()
    low = raw.lower()

    # Freetext / non-numeric → NaN
    FREETEXT = {
        'tbd', 'ud', 'to be discussed', 'monthly salary', 'basic salary',
        'monthly', 'competitive salary', 'attractive salary package',
        'attractive salary', 'competitive pay', 'world class benefits',
        'basic + medical', 'basic + allowance + commission', '$0 - $0'
    }
    if low in FREETEXT or not re.search(r'\d', raw):
        return pd.Series([np.nan, np.nan, np.nan, np.nan])

    # ── Currency detection ────────────────────────────────────────────────
    if   'IDR' in raw or re.search(r'\bRp\b', raw): currency = 'IDR'
    elif 'PHP' in raw or '₱' in raw:                currency = 'PHP'
    elif 'THB' in raw or '฿' in raw:                currency = 'THB'
    elif 'SGD' in raw:                               currency = 'SGD'
    elif 'HKD' in raw or 'HK$' in raw:              currency = 'HKD'
    elif 'AUD' in raw or 'A$' in raw:               currency = 'AUD'
    elif 'USD' in raw:                               currency = 'USD'
    elif re.search(r'\bRM\b|\bMYR\b', raw):         currency = 'MYR'
    else:                                            currency = 'Unknown'

    # ── Period detection ──────────────────────────────────────────────────
    if   re.search(r'p\.a\.|per year|yearly',  low): period = 'yearly'
    elif re.search(r'p\.h\.|per hour|hourly',  low): period = 'hourly'
    elif re.search(r'p\.m\.|per month|monthly',low): period = 'monthly'
    else:                                             period = 'unknown'

    # ── Number extraction ─────────────────────────────────────────────────
    if currency == 'IDR':
        # IDR uses dots as thousand-separators: Rp 10.000.000 → 10000000
        nums_raw = re.findall(r'\d[\d.]*', raw)
        nums = []
        for n in nums_raw:
            try: nums.append(float(n.replace('.', '')))
            except: pass
    else:
        # Expand shorthand: 10k → 10000, 1.5k → 1500
        cleaned = re.sub(
            r'(\d+(?:\.\d+)?)\s*k',
            lambda m: str(float(m.group(1)) * 1000),
            raw, flags=re.IGNORECASE
        )
        nums_raw = re.findall(r'\d[\d,]*(?:\.\d+)?', cleaned)
        nums = []
        for n in nums_raw:
            try: nums.append(float(n.replace(',', '')))
            except: pass

    nums = [n for n in nums if n > 0]  # drop zeros

    sal_min = nums[0] if len(nums) >= 1 else np.nan
    sal_max = nums[1] if len(nums) >= 2 else sal_min

    # Swap if inverted
    if pd.notna(sal_min) and pd.notna(sal_max) and sal_min > sal_max:
        sal_min, sal_max = sal_max, sal_min

    return pd.Series([sal_min, sal_max, currency, period])


df[['salary_min', 'salary_max', 'salary_currency', 'salary_period']] = df['salary'].apply(parse_salary)

print('=== Currency breakdown ===')
print(df['salary_currency'].value_counts())
print()
print('=== Period breakdown ===')
print(df['salary_period'].value_counts())
print()
print(f"Rows with parseable salary : {df['salary_min'].notna().sum():,}")
print(f"Rows with no salary info   : {df['salary_min'].isna().sum():,}")
print()
# Spot-check one row per currency
for cur in ['MYR', 'SGD', 'IDR', 'PHP', 'THB', 'USD']:
    sample = df[df['salary_currency'] == cur][['salary', 'salary_min', 'salary_max', 'salary_period']].head(1)
    if not sample.empty:
        print(f'[{cur}]', sample.to_string(index=False))


=== Currency breakdown ===
salary_currency
MYR        13373
SGD         1208
Unknown       87
USD           15
HKD            3
IDR            1
Name: count, dtype: int64

=== Period breakdown ===
salary_period
monthly    14460
hourly       106
unknown       63
yearly        58
Name: count, dtype: int64

Rows with parseable salary : 14,687
Rows with no salary info   : 13,208

[MYR]                         salary  salary_min  salary_max salary_period
RM 6,800 – RM 10,000 per month      6800.0     10000.0       monthly
[SGD]                          salary  salary_min  salary_max salary_period
$2,800 – $3,500 per month (SGD)      2800.0      3500.0       monthly
[IDR]                                        salary  salary_min  salary_max salary_period
Rp 10.000.000 – Rp 12.000.000 per month (IDR)  10000000.0  12000000.0       monthly
[USD]                      salary  salary_min  salary_max salary_period
$700 – $900 per month (USD)       700.0       900.0       monthly


This code cell defines and applies a `parse_salary` function to extract `salary_min`, `salary_max`, `salary_currency`, and `salary_period` from the `salary` column. The function handles various currency formats, shorthand notations (e.g., '5k'), and different time periods (monthly, yearly, hourly). After applying the function, it prints breakdowns of detected currencies and periods, along with counts of rows with parseable salary information, and displays sample rows for different currencies.

## Cell 12 — Clean `posted` (Date Posted)
Standardise values like `30d+ ago•Expiring`, `3d ago`, `Just posted`, etc.

This markdown cell outlines the goal of cleaning the `posted` column, which is to standardize values like '30d+ ago' into a more usable format.

In [13]:
def parse_days_ago(val):
    """Convert posted string to numeric days-ago estimate."""
    if pd.isna(val):
        return np.nan
    v = str(val).lower().strip()

    # Treat 'just posted', 'today', or minutes ('m') as 0 days
    if 'just posted' in v or 'today' in v or 'just now' in v:
        return 0

    # Handle minutes: e.g., '42m ago'
    if re.search(r'\d+\s*m', v):
        return 0

    # Handle hours: e.g., '5h ago'
    if re.search(r'\d+\s*h', v):
        return 0

    # Handle days: e.g., '30d+' or '3d'
    m = re.search(r'(\d+)\s*d\+?', v)
    if m:
        return int(m.group(1))

    return np.nan

# Re-apply the updated function
df['days_ago'] = df['posted'].apply(parse_days_ago)
df['is_expiring'] = df['posted'].astype(str).str.lower().str.contains('expiring')

print("=== Updated days_ago Null Count ===")
print(f"Remaining nulls: {df['days_ago'].isna().sum()}")
print("\nTop 10 values:")
print(df['days_ago'].value_counts().head(10))

=== Updated days_ago Null Count ===
Remaining nulls: 0

Top 10 values:
days_ago
4     1564
7     1554
8     1543
9     1529
3     1518
1     1495
0     1492
2     1431
10    1333
16    1210
Name: count, dtype: int64


This code cell defines and applies a `parse_days_ago` function to convert the `posted` string into an estimated number of days since the job was posted. It also creates a boolean column `is_expiring` based on whether the 'posted' string contains 'expiring'. It then prints the value counts for `days_ago` and the total count of expiring listings.

## Cell 13 — Validate & Clean `url`

This markdown cell marks the beginning of the validation and cleaning process for the `url` column.

In [14]:
before = len(df)
df = df[df['url'].astype(str).str.startswith('http')]
print(f'Dropped {before - len(df):,} rows with invalid URLs → {len(df):,} remaining')

Dropped 0 rows with invalid URLs → 27,895 remaining


This code cell filters the DataFrame to keep only rows where the `url` column starts with 'http', effectively dropping rows with invalid or malformed URLs. It then prints the number of rows dropped and the remaining number of rows.

## Cell 14 — Final Missing Value Summary

This markdown cell introduces the final summary of missing values after all cleaning operations have been performed.

In [15]:
print('=== Missing Values After Cleaning ===')
missing_final = df.isnull().sum()
pct_final     = (missing_final / len(df) * 100).round(2)
summary = pd.DataFrame({'Missing': missing_final, 'Pct (%)': pct_final})
print(summary[summary['Missing'] > 0] if missing_final.any() else 'None 🎉')
print(f'\n[CLEANED DATA SHAPE] → {df.shape[0]:,} rows × {df.shape[1]} columns')

=== Missing Values After Cleaning ===
                 Missing  Pct (%)
salary             13128    47.06
salary_min         13208    47.35
salary_max         13208    47.35
salary_currency    13208    47.35
salary_period      13208    47.35

[CLEANED DATA SHAPE] → 27,895 rows × 16 columns


This code cell provides a final summary of missing values in the DataFrame after all cleaning steps. It calculates the number and percentage of missing values for each column and displays only those columns that still have missing data. It also prints the final shape of the cleaned DataFrame.

## Cell 15 — Final Column Overview

This markdown cell serves as a heading for the final overview of the DataFrame's columns.

In [16]:
print('=== Final Columns ===')
print(df.columns.tolist())
df.head(5)

=== Final Columns ===
['country', 'title', 'company', 'location', 'salary', 'posted', 'employment_type', 'industry', 'url', 'salary_disclosed', 'salary_min', 'salary_max', 'salary_currency', 'salary_period', 'days_ago', 'is_expiring']


,country,title,company,location,salary,posted,employment_type,industry,url,salary_disclosed,salary_min,salary_max,salary_currency,salary_period,days_ago,is_expiring
0,Malaysia,Assistant Jewellery Sales Manager (Malaysia) 珠寶銷售副經理 (馬來西亞),Luk Fook Prestige (Malaysia) Sdn Bhd,Kuala Lumpur,NaN,30d+ ago•Expiring,Full Time,Sales,https://my.jobstreet.com/job/88167357?type=standard&ref=search-standalone&or...,False,NaN,NaN,NaN,NaN,30,True
1,Malaysia,Cashier (Malaysia) 出納員 (馬來西亞),Luk Fook Prestige (Malaysia) Sdn Bhd,Kuala Lumpur,NaN,30d+ ago,Full Time,Retail,https://my.jobstreet.com/job/88167269?type=standard&ref=search-standalone&or...,False,NaN,NaN,NaN,NaN,30,False
2,Malaysia,Assistant Jewellery Sales Supervisor (Malaysia) 珠寶銷售副主任 (馬來西亞),Luk Fook Prestige (Malaysia) Sdn Bhd,Kuala Lumpur,NaN,30d+ ago•Expiring,Full Time,Retail,https://my.jobstreet.com/job/88167220?type=standard&ref=search-standalone&or...,False,NaN,NaN,NaN,NaN,30,True
3,Malaysia,Big Data Hadoop Developer,Unison Consulting,Kuala Lumpur,NaN,30d+ ago,Full Time,Design,https://my.jobstreet.com/job/86489402?type=standard&ref=search-standalone&or...,False,NaN,NaN,NaN,NaN,30,False
4,Malaysia,Jewellery Sales Consultant (Malaysia) 珠寶銷售顧問 (馬來西亞),Luk Fook Prestige (Malaysia) Sdn Bhd,Kuala Lumpur,NaN,30d+ ago•Expiring,Full Time,Sales,https://my.jobstreet.com/job/88166982?type=standard&ref=search-standalone&or...,False,NaN,NaN,NaN,NaN,30,True


This code cell prints the list of all column names in the final cleaned DataFrame and then displays the first five rows of the DataFrame, providing a quick look at the cleaned data and the new columns created during the cleaning process.

### Final Data Structure & Quality Report
This section provides a technical summary of the cleaned dataset, including data types, missing value counts, and basic statistics for derived features.

In [17]:
print('=== 1. Schema & Data Types ===')
display(df.info())

print('\n=== 2. Numerical Summary (Derived Features) ===')
display(df[['salary_min', 'salary_max', 'days_ago']].describe().round(2))

print('\n=== 3. Final Data Sample ===')
display(df.head())

print('\n=== 4. Column List ===')
print(df.columns.tolist())

=== 1. Schema & Data Types ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27895 entries, 0 to 27894
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   country           27895 non-null  object 
 1   title             27895 non-null  object 
 2   company           27895 non-null  object 
 3   location          27895 non-null  object 
 4   salary            14767 non-null  object 
 5   posted            27895 non-null  object 
 6   employment_type   27895 non-null  object 
 7   industry          27895 non-null  object 
 8   url               27895 non-null  object 
 9   salary_disclosed  27895 non-null  bool   
 10  salary_min        14687 non-null  float64
 11  salary_max        14687 non-null  float64
 12  salary_currency   14687 non-null  object 
 13  salary_period     14687 non-null  object 
 14  days_ago          27895 non-null  int64  
 15  is_expiring       27895 non-null  bool   
dtypes: bool(2

None


=== 2. Numerical Summary (Derived Features) ===


,salary_min,salary_max,days_ago
count,14687.00,14687.00,27895.00
mean,4928.46,6493.98,12.44
std,82752.83,99401.53,8.87
min,1.00,1.00,0.00
25%,2500.00,3300.00,4.00
50%,3500.00,4500.00,11.00
75%,4500.00,6000.00,18.00
max,10000000.00,12000000.00,30.00



=== 3. Final Data Sample ===


,country,title,company,location,salary,posted,employment_type,industry,url,salary_disclosed,salary_min,salary_max,salary_currency,salary_period,days_ago,is_expiring
0,Malaysia,Assistant Jewellery Sales Manager (Malaysia) 珠寶銷售副經理 (馬來西亞),Luk Fook Prestige (Malaysia) Sdn Bhd,Kuala Lumpur,NaN,30d+ ago•Expiring,Full Time,Sales,https://my.jobstreet.com/job/88167357?type=standard&ref=search-standalone&or...,False,NaN,NaN,NaN,NaN,30,True
1,Malaysia,Cashier (Malaysia) 出納員 (馬來西亞),Luk Fook Prestige (Malaysia) Sdn Bhd,Kuala Lumpur,NaN,30d+ ago,Full Time,Retail,https://my.jobstreet.com/job/88167269?type=standard&ref=search-standalone&or...,False,NaN,NaN,NaN,NaN,30,False
2,Malaysia,Assistant Jewellery Sales Supervisor (Malaysia) 珠寶銷售副主任 (馬來西亞),Luk Fook Prestige (Malaysia) Sdn Bhd,Kuala Lumpur,NaN,30d+ ago•Expiring,Full Time,Retail,https://my.jobstreet.com/job/88167220?type=standard&ref=search-standalone&or...,False,NaN,NaN,NaN,NaN,30,True
3,Malaysia,Big Data Hadoop Developer,Unison Consulting,Kuala Lumpur,NaN,30d+ ago,Full Time,Design,https://my.jobstreet.com/job/86489402?type=standard&ref=search-standalone&or...,False,NaN,NaN,NaN,NaN,30,False
4,Malaysia,Jewellery Sales Consultant (Malaysia) 珠寶銷售顧問 (馬來西亞),Luk Fook Prestige (Malaysia) Sdn Bhd,Kuala Lumpur,NaN,30d+ ago•Expiring,Full Time,Sales,https://my.jobstreet.com/job/88166982?type=standard&ref=search-standalone&or...,False,NaN,NaN,NaN,NaN,30,True



=== 4. Column List ===
['country', 'title', 'company', 'location', 'salary', 'posted', 'employment_type', 'industry', 'url', 'salary_disclosed', 'salary_min', 'salary_max', 'salary_currency', 'salary_period', 'days_ago', 'is_expiring']


## Cell 17 — Save Cleaned Dataset

This markdown cell introduces the final step: saving the cleaned dataset.

In [18]:
df = df.reset_index(drop=True)
df.to_csv(CLEANED_PATH, index=False, encoding='utf-8-sig')
size_mb = os.path.getsize(CLEANED_PATH) / 1e6
print(f'✅ Cleaned dataset saved → {CLEANED_PATH}  ({size_mb:.1f} MB)')
print(f'Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns')

✅ Cleaned dataset saved → jobstreet_cleaned.csv  (6.9 MB)
Final shape: 27,895 rows × 16 columns


This code cell resets the DataFrame index and then saves the cleaned DataFrame `df` to a new CSV file named `jobstreet_cleaned.csv` without including the index. It also calculates and prints the size of the saved file in MB and the final shape of the DataFrame.